# LightLogger Data Processing Pipeline

## Overview

This notebook documents a full end-to-end preprocessing and analysis pipeline for the LightLogger / FLIC dataset.

The pipeline converts raw multimodal recordings (world camera + Neon eye tracking) into structured, analyzable outputs:

- Cleaned world camera videos
- Egocentric gaze mappings (Neon → World)
- Virtually foveated (retinal-coordinate) videos
- Spatial Power Density (SPD) statistics
- Grouped visualizations (per subject and across subjects)

This pipeline is hybrid:

- **Python** → orchestration, file management, batching
- **MATLAB** → vision processing, foveation, SPD computation

---

## High-Level Pipeline

1. Generate World Videos  
2. Run Egocentric Mapping  
3. Generate Virtually Foveated Videos  
4. Compute SPDs  
5. Group SPDs (Per Subject)  
6. Group SPDs (Across Subjects)  

**Each step depends on outputs from the previous stage.**

---

## Import libraries and Extensions

In [1]:
import importlib
import preprocessing_pipeline

In [ ]:
%load_ext line_profiler
%load_ext memory_profiler

# Step 0 — Data Acquisition, Transfer, and Verification

## Goal

Ensure all raw recordings are:
- Downloaded or transferred correctly
- Complete (no missing chunks/files)
- Properly paired (Neon ↔ World)
- Ready for downstream processing

This step is critical to avoid silent data corruption or misalignment later in the pipeline.

---

## What This Step Does (Detailed)

### 1. Data Download / Transfer

Steps Performed in this Stage:

- Download recordings via API (e.g., Pupil Cloud endpoints)
- Transfer `.zip` recordings from external storage
- Extract archives into the expected directory structure

Typical structure enforced:

```
FLIC_XXXX/ 
    activity/
        GKA/      (world chunks)
        Neon/     (eye tracking data)
```

### 2. File Integrity Checks

After transfer, the pipeline verifies:

- Required directories exist (`GKA`, `Neon`)
- Files are non-empty
- Expected number of chunk files is present
- No partial or truncated downloads


### 3. Video Integrity Validation

For world camera data:

- Ensures GKA chunks can be successfully read
- Verifies frame continuity (no corrupted segments)
- Confirms metadata (FPS, resolution) is consistent

This prevents downstream failures in:
- video stitching
- SPD computation

---


### Transfer Light Logger Videos to NAS

In [5]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.transfer_light_logger_recordings(subjects_to_transfer=[51],
                                                        overwrite_existing=False, 
                                                        verbose=True)

/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis/code/preprocessRecordingData/preprocessing_pipeline.py:2688: UserWarning: Recording: .Spotlight-V100 at src_dir: /Volumes/T7 Shield does not contain FLIC
  warnings.warn(f"Recording: {recording_name} at src_dir: {src_dir} does not contain FLIC")
/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis/code/preprocessRecordingData/preprocessing_pipeline.py:2688: UserWarning: Recording: .fseventsd at src_dir: /Volumes/T7 Shield does not contain FLIC
  warnings.warn(f"Recording: {recording_name} at src_dir: {src_dir} does not contain FLIC")
/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis/code/preprocessRecordingData/preprocessing_pipeline.py:2688: UserWarning: Recording: SPHERE_RECORDING at src_dir: /Volumes/T7 Shield does not contain FLIC
  warnings.warn(f"Recording: {recording_name} at src_dir: {src_dir} does not contain FLIC")
/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis/co

Processing Subjects:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/7 [00:00<?, ?it/s]

Input path: /Volumes/T7 Shield/FLIC_51_sitBiopond_temporalSensitivity_1
Output path: /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/sitBiopond/GKA
Input path: /Volumes/T7 Shield/FLIC_51_walkBiopond_temporalSensitivity_1
Output path: /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/walkBiopond/GKA
Input path: /Volumes/T7 Shield/FLIC_51_walkOutdoor_temporalSensitivity_1
Output path: /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/walkOutdoor/GKA
Input path: /Volumes/T7 Shield/FLIC_51_walkIndoor_temporalSensitivity_1
Output path: /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/walkIndoor/GKA
Input path: /Volumes/T7 Shield/FLIC_51_chat_temporalSensitivity_1
Output path: /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/chat/GKA
Input path: /Volumes/T7 Shield/FLIC_51_read_temporalSensitivity_1
Output path: /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/read/GKA
Input path: /Volumes/T7 Shield/FLIC_51_dark_temporalSe

### Download Pupil Labs Data fromt the Cloud

#### 1. Download the Neon Timeseries + Scene Videos

In [ ]:
importlib.reload(preprocessing_pipeline)
api_key: str = "" # TODO: Fill this in, but NEVER commit this. If it is committed, immediately deactivate it on Pupil Cloud
preprocessing_pipeline.download_pupil_cloud_recordings(api_key, 
                                                       dst_dir="/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026", 
                                                       subjects_to_download=[51], 
                                                       verbose=True,
                                                       overwrite_existing=False
                                                      )

/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis/code/preprocessRecordingData/preprocessing_pipeline.py:2829: UserWarning: Recording name: FLIC_2001_read_ has no recording number, Skipping...
  recording_number: int = int(re.search(r"_(\d+)$", recording_name).group(1))


Processing Subjects:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/7 [00:00<?, ?it/s]

Downloading: 51 | dark | /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/dark/Timeseries Data + Scene Video.zip
Downloading: 51 | read | /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/read/Timeseries Data + Scene Video.zip
Downloading: 51 | chat | /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/chat/Timeseries Data + Scene Video.zip
Downloading: 51 | walkIndoor | /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/walkIndoor/Timeseries Data + Scene Video.zip
Downloading: 51 | walkOutdoor | /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/walkOutdoor/Timeseries Data + Scene Video.zip
Downloading: 51 | walkBiopond | /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/walkBiopond/Timeseries Data + Scene Video.zip
Downloading: 51 | sitBiopond | /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_51/sitBiopond/Timeseries Data + Scene Video.zip


Processing Subjects:   0%|          | 0/10 [00:00<?, ?it/s]

...

Processing Activities:   0%|          | 0/7 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/9 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/7 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/9 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/7 [00:00<?, ?it/s]

### 2. Ensure downloaded neon files are well formatted and have all required content

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_neon_integrity(verbose=True)

---

# Step 1 — Generate World Camera Videos

## Goal

Convert raw chunk-based recordings into a single continuous video per activity.

## Input

Raw dataset structure:

```
FLIC_XXXX/
    activity/
        GKA/
            chunk files
```
## Output

Processed video:

```
FLIC_XXXX/
    activity/
        GKA/
            W.avi
```

## What This Step Does 

For every subject and activity:

1. Locates the `GKA` directory containing chunked frames
2. Uses `video_io.world_chunks_to_video(...)` to:
   - Stitch chunks into a continuous video stream
   - Handle missing frames via interpolation (if enabled)
   - Convert timestamps into seconds
3. Optionally applies:
   - Debayering (raw sensor → RGB)
   - Color weighting (camera calibration)
   - Digital gain correction
4. Writes a lossless `.avi` (FFV1 encoding)

---

### Generate the World Videos

In [2]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_world_videos(overwrite_existing=True, 
                                             verbose=True, 
                                             apply_digital_gain=True, 
                                             apply_color_weights=True,
                                             debayer_images=True, 
                                             apply_floor_ceiling=True,
                                             remove_dark_noise=True, 
                                             apply_fielding_function=False, 
                                             subjects_to_process=[1038],
                                             activities_to_process=["walkOutdoor"]
                                            ) 

Processing Subjects:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/1 [00:00<?, ?it/s]

Input: /Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_1038/walkOutdoor/GKA
Output: /Volumes/FLIC_processing/NEWscriptedIndoorOutdoorVideos2026/FLIC_1038/walkOutdoor/GKA/W.avi


Processing chunks:   0%|          | 0/12 [00:00<?, ?it/s]

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


...

Processing frames:   0%|          | 0/3600 [00:00<?, ?it/s]

Processing frames:   0%|          | 0/3600 [00:00<?, ?it/s]

Processing frames:   0%|          | 0/3600 [00:00<?, ?it/s]

Processing frames:   0%|          | 0/3600 [00:00<?, ?it/s]

Processing frames:   0%|          | 0/1887 [00:00<?, ?it/s]

# Step 2 — Verifying Neon ↔ World Pairing

This is one of the most important checks.

For each activity:

- Locate Neon recording directory
- Locate corresponding world video (or chunks)
- Ensure BOTH exist before proceeding

Typical logic:

```
- Neon path:
    activity/Neon/<recording_folder>
- World path:
    activity/GKA/
```

Assertions ensure:

- Both modalities exist
- Neither is empty
- They correspond to the same recording session

**When verbose is enabled, output images are displayed to visually verify the recordings are from the same session**

---



### Ensure the world videos and neon videos are correctly paired

In [ ]:
importlib.reload(preprocessing_pipeline)

# Return value is a dictionary of the recordings whose integrity was verified
videos_observed: dict[str, dict[str, bool]] = preprocessing_pipeline.verify_world_neon_pairing(verbose=True)

## 4. Run Egocentric Video Mapper
Note: you have to switch to the "egocentric_video_mapper" kernel to run this step

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_egocentric_mapper_results(overwrite_existing=True, verbose=True, subjects_to_process=[28], activities_to_process=["walkBiopond"]) # TODO: There is some bug with 28 walkBiopond

## 4. Virtually Foveate

### 1. First find the start ends of activities

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_tag_task_start_ends(overwrite_existing=False, verbose=True)

### 2. Virtually Foveate the videos

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_virtually_foveated_videos(overwrite_existing=True, verbose=True,  
                                                          activities_to_process=["chat", "walkBiopond"])

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_virtually_foveated_video_integrity(verbose=True, activities_to_skip=["phone", "lunch"])

## 5. Generate SPDs

### 1. Generate the Indinvidual SPDs and plots for all desired subjects and activitys 

In [ ]:
importlib.reload(preprocessing_pipeline)

color_modes: list[str] = ["a", "c_lm", "c_s"]
for color_mode in color_modes:
    preprocessing_pipeline.generate_spds(color_mode=color_mode,
                                        overwrite_existing=True, 
                                        verbose=True, 
                                        activities_to_process=["chat", "walkBiopond"], 
                                        )

In [ ]:
# Let's combine the SPDs for the desired colormodes 
importlib.reload(preprocessing_pipeline)

preprocessing_pipeline.combine_spds(overwrite_existing=True, 
                                    verbose=True, 
                                    activities_to_process=["chat", "walkBiopond"],
                                    color_modes_to_process=["a", "c_lm", "c_s"]
                                  )


### 2. Plot the indivudal SPDs (optionally if not done above)
We also have this as a separate optional step because the first step is ridiculously long in terms of runtime

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.adjust_spd_axes(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"], combine_figures=True)

#### 3. For each subject, output a grouped plot of their results for all activities


In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.group_spds_per_subject(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"])

#### 4. Generate average SPDs for each activity by averaging across subjects for that activity

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_subject(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                                    combine_figures=True, common_axes=True)

#### 5. Group the mean SPDs for each activity by the activity type


In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.group_spds_across_subjects(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"])

#### 6. Generate average SPDs across all activities' averages

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_all(overwrite_existing=True, verbose=True, color_mode="L-M", activities_to_skip=["gazeCalibration", "lunch", "phone"], combine_figures=True, common_axes=True)

In [ ]:
importlib.reload(preprocessing_pipeline)

sitting_vs_walking: dict = {"sitting": ["sitBiopond", "work", "chat"], 
                            "walking": ["walkBiopond", "walkIndoor", "walkOutdoor"]
                           }

indoor_vs_outdoor: dict = {"indoor": ["work", "chat", "walkIndoor"], 
                           "outdoor": ["walkOutdoor", "sitBiopond", "walkBiopond"]
                          }

preprocessing_pipeline.generate_spds_across_groups(overwrite_existing=True, 
                                                   color_mode="L-M",
                                                   verbose=True, 
                                                   activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                                   combine_figures=True, 
                                                   common_axes=True,
                                                   groups=sitting_vs_walking
                                                )

### Generate actigraphy graphs

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_actigraphy_graphs(overwrite_exsiting=True, 
                                                  verbose=True, 
                                                  subjects_to_process=[2001],
                                                  activities_to_process=["walkBiopond"]
                                                )

In [ ]:
import matlab.engine
eng = matlab.engine.start_matlab()

print(eng.version(nargout=1))